# Checkpoint A3: Intake Validation — Kiểm tra đầu vào hợp đồng

Checkpoint này triển khai hàm `validate_intake()` để kiểm tra xem một hợp đồng có đủ điều kiện để đưa vào pipeline trích xuất hay không.

**Điều kiện đạt:**
- [x] Hàm kiểm tra file không rỗng (>100 ký tự)
- [x] Phát hiện "HỢP ĐỒNG" / "hợp đồng" trong 200 ký tự đầu
- [x] Đếm số bên tham gia ("Bên A", "Bên B")
- [x] Đếm số điều khoản ("ĐIỀU" sections)
- [x] Trả về dict: is_valid, contract_type, parties, section_count, issues[]
- [x] Chạy test trên cả 3 hợp đồng mẫu

**Tiếp theo:** Sau khi pass, chuyển sang Checkpoint B3 (gọi Gemini API).

In [1]:
import re

def validate_intake(contract_text: str) -> dict:
    """Kiểm tra xem hợp đồng có đủ điều kiện để đưa vào pipeline trích xuất.

    Thuật toán kiểm tra 4 tiêu chí:
    1. Độ dài tối thiểu (>100 ký tự) — loại file rỗng hoặc quá ngắn
    2. Chứa từ khóa "HỢP ĐỒNG" trong phần đầu — xác nhận đây là hợp đồng
    3. Có ít nhất 2 bên ("Bên A", "Bên B") — hợp đồng cần 2 bên
    4. Có các "ĐIỀU" sections — cấu trúc hợp đồng tiêu chuẩn VN

    Args:
        contract_text: Nội dung đầy đủ của hợp đồng (chuỗi UTF-8)

    Returns:
        Dict chứa:
          - is_valid (bool): Đủ điều kiện để xử lý tiếp?
          - contract_type (str): Loại hợp đồng phát hiện được
          - parties (list[str]): Danh sách các bên tham gia
          - section_count (int): Số điều khoản đếm được
          - issues (list[str]): Danh sách vấn đề (rỗng nếu hợp lệ)
    """
    issues = []

    # --- Tiêu chí 1: Độ dài tối thiểu ---
    if len(contract_text) <= 100:
        issues.append("File quá ngắn (<=100 ký tự) — có thể file rỗng hoặc lỗi load")

    # --- Tiêu chí 2: Phát hiện từ khóa "HỢP ĐỒNG" trong 200 ký tự đầu ---
    header = contract_text[:200].upper()
    if "HỢP ĐỒNG" not in header:
        issues.append('Không tìm thấy "HỢP ĐỒNG" trong 200 ký tự đầu — có thể không phải hợp đồng')

    # --- Tiêu chí 3: Đếm các bên tham gia ---
    parties = []
    if re.search(r"Bên\s+A", contract_text, re.IGNORECASE):
        parties.append("Bên A")
    if re.search(r"Bên\s+B", contract_text, re.IGNORECASE):
        parties.append("Bên B")
    if re.search(r"Bên\s+C", contract_text, re.IGNORECASE):
        parties.append("Bên C")

    if len(parties) < 2:
        issues.append(f"Chỉ tìm thấy {len(parties)} bên — hợp đồng cần ít nhất 2 bên (Bên A, Bên B)")

    # --- Tiêu chí 4: Đếm điều khoản ---
    sections = re.findall(r"ĐIỀU\s+\d+", contract_text, re.IGNORECASE)
    section_count = len(sections)

    if section_count == 0:
        issues.append('Không tìm thấy "ĐIỀU" nào — cấu trúc không phải hợp đồng tiêu chuẩn VN')

    # --- Xác định loại hợp đồng ---
    contract_type = "unknown"
    text_lower = contract_text.lower()
    type_keywords = {
        "service_agreement": ["cung cấp dịch vụ", "dịch vụ viễn thông", "sla", "vận hành"],
        "lease_agreement": ["thuê", "cho thuê", "mặt bằng"],
        "purchase_contract": ["mua bán", "cung cấp thiết bị", "hàng hóa"],
        "nda": ["bảo mật thông tin", "không tiết lộ", "nda"],
    }
    for ctype, keywords in type_keywords.items():
        if any(kw in text_lower for kw in keywords):
            contract_type = ctype
            break

    # --- Kết quả ---
    is_valid = len(issues) == 0

    return {
        "is_valid": is_valid,
        "contract_type": contract_type,
        "parties": parties,
        "section_count": section_count,
        "issues": issues,
    }


# Quick test với dữ liệu mẫu
sample_valid = """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Số: HD/2026/001

Bên A: Công ty Viễn thông A
Bên B: Công ty Công nghệ B

ĐIỀU 1: Đối tượng hợp đồng
ĐIỀU 2: Thời hạn hợp đồng
ĐIỀU 3: Giá trị hợp đồng
ĐIỀU 4: Phạt vi phạm SLA
""" * 3  # Nhân lên để >100 ký tự

result = validate_intake(sample_valid)
print("Test mẫu:", result)

Test mẫu: {'is_valid': True, 'contract_type': 'service_agreement', 'parties': ['Bên A', 'Bên B'], 'section_count': 12, 'issues': []}


In [2]:
# ============================================================
# Test validate_intake trên 3 hợp đồng mẫu
# ============================================================
# Dữ liệu mô phỏng 3 hợp đồng — trong thực tế load từ file .txt

contracts = {
    "contract-001.txt": """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Số: HD/VT/2026/001

Bên A: Công ty Viễn thông Việt Nam (VNPT)
Bên B: Công ty Công nghệ Số VTI

ĐIỀU 1: Đối tượng hợp đồng
Cung cấp dịch vụ vận hành và bảo trì hệ thống mạng viễn thông.

ĐIỀU 2: Thời hạn hợp đồng
Hợp đồng có hiệu lực kể từ ngày 01 tháng 01 năm 2026 đến ngày 31 tháng 12 năm 2026.

ĐIỀU 3: Giá trị hợp đồng
Tổng giá trị: 1.800.000.000 VNĐ (Một tỷ tám trăm triệu đồng).

ĐIỀU 4: Phạt vi phạm SLA
Phạt 1% giá trị hợp đồng hàng tháng cho mỗi 0.5% giảm hiệu suất. Tối đa phạt không quá 10% giá trị hợp đồng hàng quý.

ĐIỀU 5: Bảo mật thông tin
Các bên cam kết bảo mật thông tin trong thời hạn 2 năm kể từ ngày chấm dứt hợp đồng.
""",

    "contract-002.txt": """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ
Số: HD/DV/2026/002

Bên A: Tổng Công ty Viễn thông Military
Bên B: Công ty Giải pháp ITS

ĐIỀU 1: Đối tượng hợp đồng
Cung cấp dịch vụ phát triển phần mềm quản lý.

ĐIỀU 2: Thời hạn
[ĐOẠN BỊ LỖI OCR - KHÔNG ĐỌC ĐƯỢC]

ĐIỀU 3: Giá trị
Giá trị hợp đồng: 950.000.000 VNĐ.

ĐIỀU 4: Phạt vi phạm
Phạt 0.5% cho mỗi tuần chậm tiến độ.

ĐIỀU 5: Bảo mật
[THIẾU NỘI DUNG]
""",
    # contract-002 thiếu expiry_date (Điều 2 bị lỗi OCR), thiếu bảo mật (Điều 5)

    "contract-003-risky.txt": """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Số: HD/VT/2026/003

Bên A: Công ty Viễn thông TeleCo
Bên B: Công ty Outsourcing TechPro

ĐIỀU 1: Đối tượng hợp đồng
Outsourcing vận hành mạng viễn thông.

ĐIỀU 2: Thời hạn và gia hạn
Hợp đồng có hiệu lực kể từ ngày 01 tháng 03 năm 2026 đến ngày 28 tháng 02 năm 2027.
Hợp đồng sẽ tự động gia hạn thêm 12 tháng mà không cần thông báo trước, trừ khi một bên thông báo bằng văn bản chấm dứt trước 15 ngày.

ĐIỀU 3: Giá trị hợp đồng
Tổng giá trị: 1.800.000.000 VNĐ.

ĐIỀU 4: Phạt vi phạm SLA
Phạt 5% giá trị hợp đồng hàng tháng cho mỗi giờ vượt SLA. Không giới hạn mức phạt tối đa.

ĐIỀU 5: Giới hạn trách nhiệm
Tổng trách nhiệm bồi thường của Bên B không vượt quá giá trị 01 tháng thanh toán theo hợp đồng.
""",
}

# Chạy validate trên cả 3 hợp đồng và in bảng kết quả
print("=" * 80)
print(f"{'Hợp đồng':<25} {'Hợp lệ':>7} {'Loại':<20} {'Bên':>5} {'ĐIỀU':>5} {'Issues'}")
print("=" * 80)

for name, text in contracts.items():
    result = validate_intake(text)
    valid_str = "YES" if result["is_valid"] else "NO"
    issues_str = "; ".join(result["issues"]) if result["issues"] else "-"
    print(f"{name:<25} {valid_str:>7} {result['contract_type']:<20} {len(result['parties']):>5} {result['section_count']:>5}  {issues_str}")

print("=" * 80)

# Chi tiết cho contract-003-risky (hợp đồng có rủi ro)
print("\nChi tiết contract-003-risky:")
detail = validate_intake(contracts["contract-003-risky.txt"])
for key, value in detail.items():
    print(f"  {key}: {value}")

Hợp đồng                   Hợp lệ Loại                   Bên  ĐIỀU Issues
contract-001.txt              YES service_agreement        3     5  -
contract-002.txt              YES service_agreement        2     5  -
contract-003-risky.txt        YES service_agreement        2     5  -

Chi tiết contract-003-risky:
  is_valid: True
  contract_type: service_agreement
  parties: ['Bên A', 'Bên B']
  section_count: 5
  issues: []
